# SQL: Permissões

Desde nossas primeiras aulas, realizamos conexões com o **SGBD** (Sistema de Gerenciamento de Bancos de Dados) MySQL, seja no Workbench ou pelos jupyter notebooks.

Abra o arquivo `.env` que utilizamos para configurar as variáveis de ambiente. Provavelmente você deve encontrar uma linha com este conteúdo:

`MD_DB_USERNAME="root"`

Bem, isto significa que estamos nos conectando utilizando o usuário `root`!

**<div id="user_root"></div>**
**O que é o usuário root no MySQL?**

O usuário `root` é um **superusuário** no MySQL. Ele tem acesso total ao SGBD, funcionando como um usuário administrador. Possui privilégios ilimitados para criar, modificar e excluir bancos de dados, bem como conceder ou revogar permissões a outros usuários. 

Ele é criado durante a instalação do MySQL e é altamente recomendado que sejam tomadas medidas de segurança para proteger essa conta, como definir uma senha forte e restringir o acesso remoto apenas aos endereços IP autorizados.

## E a aplicação em produção?!

Estamos desenhando soluções de banco de dados que um dia serão entregues a algum cliente (estarão em produção, prontas para serem integradas a outros sistemas). Quando a conexão com o SGBD for exposta para uso por alguma API (como a que desenvolveram no projeto), vamos precisar de um usuário e senha para conexão com o MySQL.

A resposta em <a href="#user_root">"O que é o usuário root no MySQL?"</a> já dá uma ideia que utilizar o usuário `root` nestas situações não parece correto. Caso ocorra algum vazamento da senha ou *SQL injection*, a base poderia ficar exposta tanto para leitura quanto para escrita, ou seja, um hacker poderia tanto **ver os dados** nas tabelas quanto **editar** e **exluir**.

## Qual a alternativa?!

Uma alternativa mais adequada envolve:

- **Criar usuários únicos**: vamos criar múltiplos usuário com acesso ao banco. Cada indivíduo ou aplicação em *deploy* terá seu usuário e senha personalizado (sem compartilhamento de senhas - Netflix vibes!)


- **Dar permissões**: vamos adicionar ao usuário apenas as permissões necessárias para que seu papel seja cumprido. Por exemplo, um colaborador de uma consultoria externa que presta serviços à uma empresa precisa ter acesso à base toda? Não! Ele provavelmente precisará apenas ler dados e não escrever, além de ser indicado que tenha permissão de visualização apenas em parte das tabelas (as que forem úteis para desenvolvimento do projeto).

Antes de praticar, vamos importas as bibliotecas necessárias

In [ ]:
import mysql.connector
from functools import partial
import os
import insperautograder.jupyter as ia
from dotenv import load_dotenv

load_dotenv(override=True)

def run_query(connection, query, args=None):
    with connection.cursor() as cursor:
        print("Executando query:")
        for result in cursor.execute(query, multi=True):
            if result.with_rows:
                for row in result.fetchall():
                    print(row)
            else:
                print(f"{result.rowcount} linhas afetadas.")

def get_connection_helper():

    connection = mysql.connector.connect(
        host=os.getenv("MD_DB_SERVER"),
        user=os.getenv("MD_DB_USERNAME"),
        password=os.getenv("MD_DB_PASSWORD"),
        database="fiscamuni",
    )
    return connection, partial(run_query, connection)

## Criar usuário no MySQL

Podemos criar usuário no MySQL com a seguinte sintaxe:

```MySQL
CREATE USER '<user>'@'<host>' IDENTIFIED BY '<password>';
```

Vamos experimentar! Abra o Workbench e execute:

```MySQL
CREATE USER 'joao'@'localhost' IDENTIFIED BY 'abc123';
```

Então, vamos tentar fazer uma conexão com o novo usuário. A dinâmica da aula será utilizar o Workbench com o usuário `root` e os notebooks com os usuário criados para testar as permissões.

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Provavelmente você verá em alguns materiais a indicação do uso do comando:

```SQL
FLUSH PRIVILEGES;
```

<br>

após alterar permissões de um usuário. Ele força a atualização das permissões pelo servidor.

Os `GRANTS` tomam efeito assim que executados, mas o uso do `FLUSH` pode ser necessário em alguns casos.

</div>

In [ ]:
conn1 = mysql.connector.connect(
    host="localhost",
    user="joao",
    password="abc123",
    database="musica"
)

Perceba que a conexão falhou! Isto porque, apesar do usuário ter sido criado, ele não possui as permissões necessárias!

## Dar Permissão

Para alterar permissões de um usuário, iremos utilizar `GRANT`.

A seguinte sintaxe resumida pode ser utilizada:

```mysql
GRANT PRIVILEGE ON database.table TO 'username'@'host';
```

Veja mais em https://dev.mysql.com/doc/refman/8.0/en/grant.html

Abra o Workbench e execute:

```mysql
GRANT SELECT ON musica.* TO 'joao'@'localhost';
```

Assim, o usuário terá a permissão de **SELECT** em todas as tabelas do database `musica`. Vamos testar!

In [ ]:
conn1 = mysql.connector.connect(
    host="localhost",
    user="joao",
    password="abc123",
    database="musica"
)

In [ ]:
run_query(conn1, "SELECT * FROM AUTOR")

In [ ]:
run_query(conn1, "SELECT * FROM CD")

Pronto! Agora a conexão está funcionando! Vamos fechar a conexão...

In [ ]:
# Não vamos manter a conexão aberta!
conn1.close()

... e tentar realizar a conexão com a base de dados `tranqueira`:

In [ ]:
conn2 = mysql.connector.connect(
    host="localhost",
    user="joao",
    password="abc123",
    database="tranqueira"
)

Obtivemos novamente um erro. Caso o usuário `joao` realmente necessite acesso à base `tranqueira`, um novo `GRANT` deve ser realizado!

Ao fazer *deploy* de aplicações em produção, é indicado ter usuários diferentes por aplicação, apenas as permissões necessárias.

Vamos criar um segundo usuário com permissão de `SELECT` na base `tranqueira`. Execute o comando no Workbench:
    
```mysql
CREATE USER 'ana'@'localhost' IDENTIFIED BY '456456';
GRANT SELECT ON tranqueira.* TO 'ana'@'localhost';
```
Teste a conexão:

In [ ]:
conn3 = mysql.connector.connect(
    host="localhost",
    user="ana",
    password="456456",
    database="tranqueira"
)

Vamos ver se conseguimos ler a tabela `perigo`:

In [ ]:
run_query(conn3, "SELECT * FROM perigo")

E fazer um **INSERT**, também na tabela `perigo`:

In [ ]:
sql = """
INSERT INTO `tranqueira`.`perigo`
    (`id`,`Nome`)
VALUES
    (10,'Moral')
"""

run_query(conn3, sql);

Novamente, precisamos dar permissão de inserção! Dê a permissão pelo Workbench e teste novamente! Perceba que agora a permissão será para **apenas uma tabela**:

```mysql
GRANT INSERT ON tranqueira.perigo TO 'ana'@'localhost';
```

In [ ]:
conn3.rollback()

In [ ]:
sql = """
INSERT INTO `tranqueira`.`perigo`
    (`id`, `Nome`)
VALUES
    (10,'Moral')
"""

run_query(conn3, sql);

Vamos verificar se realmente conseguimos inserir:

In [ ]:
run_query(conn3, "SELECT * FROM perigo")

Então, desfazemos a inserção e fechamos a conexão!

In [ ]:
conn3.rollback()
conn3.close()

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Algums exemplos de GRANTS!
    
```mysql
GRANT ALL PRIVILEGES ON *.* TO 'maria'@'localhost' WITH GRANT OPTION;
GRANT CREATE TEMPORARY TABLES ON coemu.* TO 'user_deploy_api'@'localhost';
GRANT EXECUTE ON sys.* TO 'user_dashboard'@'localhost' WITH GRANT OPTION;
GRANT SELECT, SHOW VIEW ON cartracking.* TO 'leitor'@'localhost';
```
</div>

## Revogar Permissão

Para ver as permissões de um usuário, utilize:

```mysql
SHOW GRANTS FOR 'ana'@'localhost';
```


Caso queira revogar permissões de um usuário, iremos utilizar `REVOKE`.

```mysql
REVOKE SELECT ON tranqueira.* FROM 'ana'@'localhost';
```

In [ ]:
conn4 = mysql.connector.connect(
    host="localhost",
    user="ana",
    password="456456",
    database="tranqueira"
)

Conseguimos fazer o login. Vamos testar se o **SELECT** funciona!

In [ ]:
run_query(conn4, "SELECT * FROM perigo;")

Vamos analisar os grants restantes ao usuário:

In [ ]:
run_query(conn4, "SHOW GRANTS FOR 'ana'@'localhost';")

Temos o `GRANT INSERT` mas não mais o `GRANT SELECT`, então nossa única permissão é para inserir linhas na tabela `perigo`.

In [ ]:
conn4.close()

Remova também esta permissão:

```mysql
REVOKE INSERT ON tranqueira.perigo FROM 'ana'@'localhost';
```

E teste novamente:

In [ ]:
conn5 = mysql.connector.connect(
    host="localhost",
    user="ana",
    password="456456",
    database="tranqueira"
)

## Hosts

Ao criar um usuário no MySQL, podemos especificar de qual **host** ele pode se conectar. Nos exemplos que apresentamos, o usuário `joao` só poderá se conectar ao MySQL do `localhost`.

Para permitir que o usuário se conecte de outros hosts, devemos criar um novo usuário com o mesmo nome e senha, mas com uma configuração de `host` diferente.

Por exemplo, para permitir que o usuário `joao` se conecte de qualquer host, você pode criar um novo usuário com a seguinte instrução SQL:

```mysql
CREATE USER 'joao'@'%' IDENTIFIED BY 'abc123';
```

No exemplo acima, o símbolo `%` indica que o usuário pode se conectar de qualquer host.

Então, basta condeder as permissões adequadas. Se quiser todas as permissões para o usuário "joao" ao banco de dados `tranqueira`, execute a seguinte instrução SQL:

```mysql
GRANT ALL PRIVILEGES ON tranqueira.* TO 'joao'@'%';
```

## Exercícios para entrega

Esta aula tem atividade para entrega, confira os prazos e exercícios

In [ ]:
ia.tasks()

In [ ]:
ia.grades(task="permissions")

## Base de dados

Utilizaremos a base de dados `fiscamuni`, que busca armazenar informações sobre fiscais e multas aplicadas à propriedades pertecentes a uma empresa.

A base possui o seguinte modelo relacional:

<img src="img/eer_diagram.png">

Execute o script `multas.sql` no Workbench para gerar a base.

Vamos criar nossa tradicional conexão!

In [ ]:
root_connection, db = get_connection_helper()

**Exercício 1**:

Crie um usuário `camilaw2` com login a partir do `localhost` e senha `699a1deacb58`.

In [ ]:
sql_ex01 = """
CREATE USER 'camilaw2'@'localhost' IDENTIFIED BY '699a1deacb58'
"""

db(sql_ex01)

In [ ]:
ia.sender(answer="sql_ex01", task="permissions", question="ex01", answer_type="pyvar")

**Exercício 2**:

Remova o usuário criado no exercício anterior.

In [ ]:
sql_ex02 = """
DROP USER 'camilaw2'@'localhost'
"""

db(sql_ex02)

In [ ]:
ia.sender(answer="sql_ex02", task="permissions", question="ex02", answer_type="pyvar")

**Exercício 3**:

Crie um usuário `marianafag` com login a partir de **qualquer host** e senha `cB18cDd2503F`.

In [ ]:
sql_ex03 = """
CREATE USER 'marianafag'@'%' IDENTIFIED BY 'cB18cDd2503F'
"""

db(sql_ex03)

In [ ]:
ia.sender(answer="sql_ex03", task="permissions", question="ex03", answer_type="pyvar")

**Exercício 4**:

Crie um usuário `pereiradjs` com login a partir do IP `192.168.15.160` e senha `bb3_091#2d6@A70`.

In [ ]:
sql_ex04 = """
CREATE USER 'pereiradjs'@'192.168.15.160' IDENTIFIED BY 'bb3_091#2d6@A70'
"""

db(sql_ex04)

In [ ]:
ia.sender(answer="sql_ex04", task="permissions", question="ex04", answer_type="pyvar")

**Exercício 5**:

Utilize SQL para criar um usuário, observando que:

- O nome de usuário será `diniz`
- A senha será `abc123cba`
- Deve conseguir login de qualquer host

Ainda, o usuário deve ter permissões apenas de:

- **Leitura** na tabela `fiscal`. 

Ou seja, inserção e deleção ou select em outras tabelas ou bases de dados devem estar bloqueadas.

In [ ]:
sql_ex05 = """
CREATE USER 'diniz'@'%' IDENTIFIED BY 'abc123cba';
GRANT SELECT ON fiscamuni.fiscal TO 'diniz'@'%';
"""

db(sql_ex05)

In [ ]:
ia.sender(answer="sql_ex05", task="permissions", question="ex05", answer_type="pyvar")

**Exercício 6**:

Crie um usuário `rem_dash_alu` com login a partir de IPs da **subnet /24** em `192.168.58.0` com senha `9C26189563A7`.

In [ ]:
sql_ex06 = """
CREATE USER 'rem_dash_alu'@'192.168.58.%' IDENTIFIED BY '9C26189563A7'
"""

db(sql_ex06)

In [ ]:
ia.sender(answer="sql_ex06", task="permissions", question="ex06", answer_type="pyvar")

**Exercício 7**:

Suponha que um exista um usuário `marianatt` com permissão de login de **qualquer host**.

Altere a senha do usuário para `b2a8b85f76b1b923`.

**Atenção**: caso queira testar localmente, crie antes o usuário no **Workbench**.

In [ ]:
sql_ex07 = """
ALTER USER 'marianatt'@'%' IDENTIFIED BY 'b2a8b85f76b1b923'
"""

db(sql_ex07)

In [ ]:
ia.sender(answer="sql_ex07", task="permissions", question="ex07", answer_type="pyvar")

**Exercício 8**:

Escreva uma query que liste:

- O `id_empresa` da empresa.
- A quantidade de propriedades que a empresa possui cadastrada na base. Aqui, a coluna deve se chamar `qtde_propriedades`.
- O valor total de multas da empresa. Aqui, a coluna deve se chamar `valor_total_multas`.

**Obs**:
- Empresas sem propriedades devem ser retornadas com valor zerado (`0`) em `qtde_propriedades`.
- Empresas sem multas devem ser retornadas com valor zerado (`0.00`) em `valor_total_multas`.

Exiba ordenado por:
- `valor_total_multas` (decrescente) e `qtde_propriedades` (decrescente)

In [ ]:
sql_ex08 = """
SELECT
    e.id_empresa,
    COUNT(DISTINCT p.id_propriedade) AS qtde_propriedades,
    COALESCE(SUM(m.valor), 0.00) AS valor_total_multas
FROM empresa e
    LEFT OUTER JOIN propriedade p ON p.id_empresa = e.id_empresa
    LEFT OUTER JOIN multa m ON m.id_propriedade = p.id_propriedade
GROUP BY e.id_empresa
ORDER BY valor_total_multas DESC, qtde_propriedades DESC
"""

db(sql_ex08)

In [ ]:
ia.sender(answer="sql_ex08", task="permissions", question="ex08", answer_type="pyvar")

**Exercício 9**:

Transforme a query do exercício anterior em uma **view** chamada `abt_empresa_total`.

Envie apenas o comando de criação da view, não envie `DROP`!

In [ ]:
sql_ex09 = """
CREATE VIEW abt_empresa_total AS
SELECT
    e.id_empresa,
    COUNT(DISTINCT p.id_propriedade) AS qtde_propriedades,
    COALESCE(SUM(m.valor), 0.00) AS valor_total_multas
FROM empresa e
    LEFT OUTER JOIN propriedade p ON p.id_empresa = e.id_empresa
    LEFT OUTER JOIN multa m ON m.id_propriedade = p.id_propriedade
GROUP BY e.id_empresa
ORDER BY valor_total_multas DESC, qtde_propriedades DESC
"""

db(sql_ex09)

In [ ]:
ia.sender(answer="sql_ex09", task="permissions", question="ex09", answer_type="pyvar")

**Exercício 10**:

Sua empresa contratou uma consultoria para atuar em um projeto.

Relações entre empresas sempre expoem problemas de confiança, onde uma não quer que a outra tenha acesso a todos os seus dados.

Assim:
- Crie um usuário `caiomc_consult` com permissão de login de **qualquer host** e senha `6b7997f42e0ebf3a51d2`.
- O consultor deve ter permissão **apenas** de **LEITURA** na **view** `abt_empresa_total`.

Envia todas as queries em uma única string, separadas por `;`

In [ ]:
sql_ex10 = """
CREATE USER 'caiomc_consult'@'%' IDENTIFIED BY '6b7997f42e0ebf3a51d2';
GRANT SELECT ON fiscamuni.abt_empresa_total TO 'caiomc_consult'@'%';
"""

db(sql_ex10)

In [ ]:
ia.sender(answer="sql_ex10", task="permissions", question="ex10", answer_type="pyvar")

**Exercício 11**:

Foi criada uma aplicação que captura dados e faz a ingestão deles na base.

A política aceita pela empresa é de que a aplicação deve ter as permissões de:

- **INSERIR** e **LER** das tabelas da base `fiscamuni`:
    - empresa
    - fiscal
    - multa
    - propriedade
- **LER** das tabelas e views da base `fiscamuni`:
    - abt_empresa_total
    - motivo

Considere que a aplicação utiliza um usuário `u_ingest_multa` já existente com permissão de login de **qualquer host** e senha `e7854285319f1c83fcd1`.

Envia todas as queries em uma única string, separadas por `;`

In [ ]:
sql_ex11 = """
GRANT INSERT, SELECT ON fiscamuni.empresa TO 'u_ingest_multa'@'%';
GRANT INSERT, SELECT ON fiscamuni.fiscal TO 'u_ingest_multa'@'%';
GRANT INSERT, SELECT ON fiscamuni.multa TO 'u_ingest_multa'@'%';
GRANT INSERT, SELECT ON fiscamuni.propriedade TO 'u_ingest_multa'@'%';
GRANT SELECT ON fiscamuni.abt_empresa_total TO 'u_ingest_multa'@'%';
GRANT SELECT ON fiscamuni.motivo TO 'u_ingest_multa'@'%';
"""

db(sql_ex11)

In [ ]:
ia.sender(answer="sql_ex11", task="permissions", question="ex11", answer_type="pyvar")

**Exercício 12**:

Considere que o usuário `u_ingest_multa` tem todas as permissões relatadas no exercício anterior.

Ele não irá mais necessitar **INSERIR** na tabela `empresa`, nem **LER**/**INSERIR** na tabela `fiscal`.

Faça as atualizações, revogando as permissões não mais necessárias.

In [ ]:
sql_ex12 = """
REVOKE INSERT ON fiscamuni.empresa FROM 'u_ingest_multa'@'%';
REVOKE SELECT, INSERT ON fiscamuni.fiscal FROM 'u_ingest_multa'@'%';
"""

db(sql_ex12)

In [ ]:
ia.sender(answer="sql_ex12", task="permissions", question="ex12", answer_type="pyvar")

**Exercício 13**:

O consultor (o mesmo dos exercícios anteriores) necessita de acesso a mais dados.

Porém, a empresa não quer liberar o acesso completo à base (contento o nome dos fiscais, nome das propriedades multadas e demais informações sensíveis).

**a)** Que solução você utilizaria neste caso?

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Criar **views** que exponham apenas o necessário e conceder ao consultor permissão de leitura **somente sobre essas views**, nunca sobre as tabelas-base.

A view funciona como uma camada de abstração: ela pode omitir colunas sensíveis (nome do fiscal, endereço da propriedade), filtrar linhas, e apresentar apenas identificadores ou dados já agregados. Como o `GRANT SELECT` é dado na view e não na tabela, o consultor não consegue consultar a tabela original nem descobrir o que foi omitido.

Complementarmente, pode-se aplicar **anonimização/pseudonimização** nas colunas que ainda precisam ser distinguíveis (ex.: hash das descrições), como será feito no item **b)**.

</div>

**b)** Vamos supor que retornar os IDs não seja suficiente. Isto ocorre, por exemplo, quando você quer retornar o endereço de alguém, permitindo que o analista identifique que são endereços diferentes, mas sem saber exatamente qual rua.

Note que podemos ter muitas pessoas com o mesmo endereço(ex: mesma rua), e todas devem estar com o mesmo valor no campo.

Para este caso, uma sugestão é aplicar uma função de HASH, como SHA256.

Crie uma **view** `propriedade_consult` que contenha o `id` e o SHA256 da `descricao`, `cidade` e `endereco` da tabela `propriedade`. Mantenha o nome original das colunas.

Assim, o usuário utilizado pela consultoria poderia ter permissão de leitura apenas na **view** `propriedade_consult` e não na tabela original. Esta parte não precisa fazer, se quiser, teste localmente! 

In [ ]:
sql_ex13b = """
CREATE VIEW propriedade_consult AS
SELECT
    id_propriedade,
    SHA2(descricao, 256) AS descricao,
    SHA2(cidade, 256) AS cidade,
    SHA2(endereco, 256) AS endereco
FROM propriedade
"""

db(sql_ex13b)

In [ ]:
ia.sender(answer="sql_ex13b", task="permissions", question="ex13b", answer_type="pyvar")

**Exercício 14**:

Considere o retorno de uma coluna de **CPF** uma tabela utilizando SHA256.

Por exemplo:
```mysql
SELECT SHA2('377.662.560-02', 256)
UNION
SELECT SHA2('404.483.920-46', 256)
UNION
SELECT SHA2('196.499.400-49', 256)
UNION
SELECT SHA2('895.322.380-69', 256);
```

Analise a seguinte afirmação: Tendo os hashs e sabendo que é um campo de CPF, é impossível descobrir os CPFs originais.

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

A afirmação é **falsa**.

SHA256 é uma função determinística e sem chave: o mesmo CPF gera sempre o mesmo hash. Como o domínio de CPFs é **pequeno e totalmente enumerável** (10^11 combinações de 11 dígitos, e menos ainda considerando os dois dígitos verificadores — cerca de 10^9 CPFs válidos), um atacante pode simplesmente gerar o hash de todos os CPFs possíveis e comparar (ataque de **força bruta** / **rainbow table**). Em hardware comum isso leva minutos, pois SHA256 é propositalmente rápida.

Ou seja, saber que o campo é um CPF elimina a proteção: o hash vira apenas uma tabela de tradução reversível por busca exaustiva.

Mitigações: usar **salt** por registro (impede rainbow tables pré-computadas), usar funções **lentas** e com fator de custo (bcrypt, scrypt, Argon2) ou, quando o objetivo é apenas distinguir registros, usar um **HMAC com chave secreta** mantida fora do banco — sem a chave, o atacante não consegue enumerar.

</div>

**Exercício 15**:

Pesquise sobre anonimização de dados e mascaramento de dados. Explique a importância e como funciona.

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

**Anonimização** é o processo de remover ou transformar dados de forma **irreversível**, de modo que o titular não possa mais ser identificado, nem direta nem indiretamente. Dados efetivamente anonimizados deixam de ser dados pessoais e saem do escopo da LGPD/GDPR. Técnicas: supressão de colunas, generalização (trocar a data de nascimento pela faixa etária, o endereço pela cidade), agregação, *k-anonimato*, *l-diversidade* e privacidade diferencial (adicionar ruído estatístico).

**Mascaramento** é a substituição dos valores reais por valores fictícios porém com o **mesmo formato**, preservando a usabilidade do dado. Pode ser estático (uma cópia mascarada para ambientes de desenvolvimento/teste) ou dinâmico (o mascaramento acontece na consulta, conforme o perfil do usuário — ex.: `***.662.***-**`). Diferentemente da anonimização, costuma ser reversível ou pelo menos preservar a referência ao dado original.

**Importância**:
- **Conformidade legal**: LGPD, GDPR e similares exigem minimização de dados e limitação de acesso a dados pessoais.
- **Redução da superfície de risco**: um vazamento de base mascarada/anonimizada tem impacto muito menor.
- **Viabiliza o uso secundário dos dados**: permite entregar bases a consultorias, times de analytics, treinamento de modelos e ambientes de teste sem expor titulares.
- **Princípio do menor privilégio**: cada perfil enxerga apenas o que precisa para seu trabalho.

**Cuidado**: anonimização mal feita é reversível por **ataque de ligação** (cruzamento com bases externas). Remover o nome não basta se o restante (CEP + data de nascimento + sexo) identifica a pessoa — é preciso avaliar os quase-identificadores em conjunto.

</div>

**Exercício 16**:

Esqueça o usuário do banco de dados nesta questão, uma vez que ele é o usuário utilizado pelas aplicações em deploy e engenheiros da empresa.

Suponha que você foi contratado para criar uma aplicação que necessita de **login**. Os usuário devem possuir, pelo menos os campos de `id`, `nickname` e `senha`.

**a)** Construa a DDL de criação da tabela de `usuario`.

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

```mysql
CREATE TABLE usuario (
    id INT NOT NULL AUTO_INCREMENT,
    nickname VARCHAR(50) NOT NULL,
    senha_hash CHAR(60) NOT NULL,
    criado_em DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (id),
    UNIQUE KEY uk_usuario_nickname (nickname)
);
```

A senha **não** é armazenada em uma coluna `senha`, e sim em `senha_hash`: um `CHAR(60)`, tamanho exato do hash bcrypt (que já embute o algoritmo, o fator de custo e o salt na própria string).

</div>

**b)** Qual seria a query para realizar uma inserção de um usuário nesta tabela? Utilize dados fictícios!

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

```mysql
INSERT INTO usuario (nickname, senha_hash)
VALUES ('miguel.damasio', '$2b$12$3Qw8kZ1vYl0nJhQeRbF9xO7pS2uTgKx5mWc4dN6aLpE8rVzHtYbCu');
```

O valor inserido é o hash **bcrypt** (fator de custo 12) gerado pela aplicação; a senha em texto puro nunca trafega para o banco nem aparece na query.

</div>

**c)** Como você armazenou a senha no exercício "b)"? Você deixou como *plain-text*? Se sim, explique se foi uma boa escolha e quais as consequências!

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Não, a senha **não** foi armazenada como *plain-text* — foi gravado o hash bcrypt gerado pela aplicação antes do `INSERT`.

Armazenar em *plain-text* seria uma escolha ruim. Consequências:

- **Vazamento total e imediato**: qualquer dump da base, backup extraviado, SQL injection ou acesso indevido de um DBA/colaborador expõe todas as senhas prontas para uso.
- **Ataque de *credential stuffing***: como as pessoas reutilizam senhas, as credenciais vazadas seriam testadas em e-mail, bancos e redes sociais dos usuários — o dano ultrapassa a aplicação.
- **Impossibilidade de negar autoria**: sem hash, nem a própria empresa consegue provar que não usou/alterou a senha do usuário.
- **Não conformidade**: viola a LGPD (art. 46 — medidas de segurança adequadas) e boas práticas como o OWASP ASVS, com risco de sanção e dano reputacional.

Ninguém, nem o administrador do sistema, deve ser capaz de ler a senha de um usuário.

</div>

**d)** Quais seriam alternativas melhores para armanenar dados sensíveis (como senhas) em banco de dados?

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

O princípio é: senha **nunca** se guarda de forma recuperável — guarda-se um verificador que só confirma se a senha apresentada confere.

- **Funções de hash lentas e próprias para senha**, com salt único por usuário e fator de custo ajustável: **Argon2id** (recomendação atual do OWASP), **scrypt** ou **bcrypt**. O custo alto torna a força bruta economicamente inviável e pode ser aumentado conforme o hardware evolui.
- **Evitar hashes rápidos de propósito geral** (MD5, SHA-1, SHA-256 puros): foram feitos para velocidade, permitindo bilhões de tentativas por segundo em GPU.
- **Salt** aleatório e único por registro, armazenado junto ao hash — inutiliza rainbow tables e impede identificar que dois usuários têm a mesma senha.
- **Pepper**: segredo global guardado **fora do banco** (variável de ambiente, HSM, cofre como Vault/KMS); sem ele, um dump da base isolado não permite ataque offline.
- **Criptografia com chave gerenciada (KMS/HSM)** para dados sensíveis que precisam ser **recuperados** (CPF, cartão) — para senha, não: hash é preferível justamente por ser irreversível.
- **Controles de apoio**: MFA, rate limiting e bloqueio progressivo no login, política de senha baseada em tamanho, e verificação contra listas de senhas já vazadas (ex.: *Have I Been Pwned*).
- **No banco**: TLS em trânsito, criptografia em repouso, e o princípio do menor privilégio visto nesta aula — a aplicação não precisa de `root`.

</div>

**e)** Pesquise sobre *Salting & peppering passwords*. Explique como funciona!

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

Ambos adicionam um valor extra à senha antes do hash, mas com propósitos e locais de armazenamento diferentes.

**Salting** — um valor aleatório, **único por usuário**, concatenado à senha antes de aplicar a função de hash:

```
hash = H(salt || senha)
```

O salt é armazenado **junto ao hash**, em texto claro (no bcrypt ele já vem embutido na string do hash). Ele não é secreto. Serve para:
- **Inutilizar rainbow tables**: tabelas pré-computadas não servem, pois cada usuário tem um salt diferente.
- **Forçar ataque individualizado**: quebrar 1.000 senhas custa 1.000 vezes mais, em vez de uma única passada sobre toda a base.
- **Esconder senhas repetidas**: dois usuários com a mesma senha produzem hashes diferentes.

**Peppering** — um valor secreto **global** (o mesmo para toda a aplicação), também misturado à senha, mas guardado **fora do banco de dados**: em variável de ambiente, arquivo de configuração protegido, HSM ou cofre de segredos.

```
hash = H(pepper || salt || senha)
```

ou, preferencialmente, aplicando um **HMAC** com o pepper como chave. Como o pepper não está na base, um atacante que obtenha **apenas** um dump do banco não consegue nem sequer iniciar o ataque offline de força bruta — falta-lhe um segredo que nunca esteve ali.

**Na prática**: salt é obrigatório e já vem de graça em bcrypt/scrypt/Argon2; pepper é uma camada adicional de defesa em profundidade, útil quando o risco principal é o vazamento do banco isolado. O ponto de atenção do pepper é a **rotação**: como é global, trocá-lo exige reprocessar os hashes (normalmente no próximo login bem-sucedido do usuário).

</div>

**f)** Pesquise sobre *senhas e entropia*. Anote abaixo os principais aprendizados!

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">

**Entropia** mede a imprevisibilidade de uma senha — quantas tentativas um atacante precisaria, em média, para adivinhá-la. É expressa em **bits**: *n* bits significam 2^n possibilidades.

Para uma senha gerada aleatoriamente:

```
entropia = comprimento × log2(tamanho do alfabeto)
```

Ex.: 8 caracteres de um alfabeto de 94 símbolos imprimíveis ≈ 8 × 6,55 ≈ **52 bits**; 16 caracteres ≈ **105 bits**.

Principais aprendizados:

- **Comprimento vale mais que complexidade**: a entropia cresce **linearmente** com o comprimento e apenas **logaritmicamente** com o tamanho do alfabeto. Acrescentar um caractere rende mais do que exigir mais um símbolo especial.
- **A fórmula só vale para senhas realmente aleatórias**. `P@ssw0rd!` tem 9 caracteres e alfabeto grande, mas entropia real baixíssima: é uma palavra de dicionário com substituições previsíveis, e ataques por dicionário com regras a quebram instantaneamente. O que importa é a entropia **da forma como a senha foi escolhida**, não a aparência do resultado.
- **Senhas escolhidas por humanos têm entropia muito baixa** (estimativas em torno de 20–30 bits), porque seguem padrões: nome + ano, maiúscula no início, número e símbolo no fim.
- **Passphrases** (método *diceware*) são um ótimo equilíbrio: 6 palavras sorteadas de uma lista de 7.776 rendem ≈ 77 bits, sendo muito mais fáceis de memorizar que uma string aleatória equivalente.
- **Referência prática**: abaixo de ~40 bits é fraco; ~60–80 bits é razoável para contas comuns; acima de ~100 bits é adequado para chaves e contas críticas.
- **A entropia define o custo do ataque offline**, mas o custo por tentativa vem do algoritmo de hash. Por isso ela é **complementar** ao uso de Argon2/bcrypt, salt e pepper — e nenhuma entropia salva uma senha reutilizada em um serviço que já vazou.
- **Consequência para políticas de senha**: as diretrizes atuais (NIST SP 800-63B) recomendam exigir **comprimento mínimo** e checagem contra listas de senhas vazadas, em vez de regras de composição obrigatória e expiração periódica, que empurram o usuário para padrões previsíveis — ou seja, reduzem a entropia real.

</div>

### Fechando a conexão

In [ ]:
root_connection.close()

## Conferir Notas

Confira se as notas na atividade são as esperadas!

In [ ]:
ia.grades(by="task", task="permissions")

In [ ]:
ia.grades(task="permissions")

In [ ]:
ia.grades(by="task")

Média de ATV:

In [ ]:
# Média de ATV, dividindo por n-2
ia.average(excluded_count=2)

Até a próxima aula!